<a href="https://colab.research.google.com/github/claudiorafaels/nondeterminism_llm_inference/blob/main/labs_2026_04_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1
!pip install vllm sentence-transformers

In [ ]:
from huggingface_hub import login
from google.colab import userdata

meu_token = userdata.get('HF_TOKEN')
login(token=meu_token)

print("Autenticado com sucesso via Secrets!")

In [ ]:
# Célula 3: Motor 7B
%env VLLM_USE_V1=0
%env VLLM_WORKER_MULTIPROC_METHOD=spawn

from vllm import LLM, SamplingParams

print("Iniciando Qwen 7B na GPU A100 (BFloat16 + Deep Layers)...")

try:
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct",
        dtype="bfloat16",
        max_model_len=1024,
        gpu_memory_utilization=0.8,
        enforce_eager=False,
        distributed_executor_backend="uni"
    )
    print("\n✅ Motor de 7B carregado! VRAM alocada com sucesso.")
except Exception as e:
    print(f"\n❌ Falha: {e}")

In [ ]:
# Célula 4/5 V4: Simulador de Sensibilidade por Consistência Interna
import random
import numpy as np
import pandas as pd
from collections import Counter
from vllm import SamplingParams

sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=10, seed=42)
prompt_bruto = "Defina o conceito de Transformação Digital em exatamente 6 palavras, nem mais, nem menos."
prompt_formatado = llm.get_tokenizer().apply_chat_template([{"role": "user", "content": prompt_bruto}], tokenize=False, add_generation_prompt=True)

grupos_carga = [int(x) for x in np.linspace(1, 250, 30)]
iteracoes_por_grupo = 20
resultados_estatisticos = []

print(f"Iniciando Teste Escalonado por Consistência Interna ({len(grupos_carga)} grupos, 600 inferências totais)...\n")

for carga in grupos_carga:
    respostas_deste_grupo = []

    print(f"🚀 Testando patamar de {carga} requisições simultâneas...")

    for i in range(iteracoes_por_grupo):
        # Apenas simulamos o ruído se a carga for maior que 1
        if carga > 1:
            prompts_ruido = [f"Explain tech in {random.randint(10, 500)} words."] * (carga - 1)
            lote = [prompt_formatado] + [
                llm.get_tokenizer().apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                for p in prompts_ruido
            ]
        else:
            lote = [prompt_formatado]

        out = llm.generate(lote, sampling_params, use_tqdm=False)
        # O resultado do prompt alvo é sempre o índice 0 do lote
        texto_gerado = out[0].outputs[0].text.strip()
        respostas_deste_grupo.append(texto_gerado)

    # Contar ocorrências de cada resposta gerada nestas 20 iterações
    contagem = Counter(respostas_deste_grupo)
    qtd_respostas_unicas = len(contagem)

    # A resposta dominante (Moda) será considerada a "correta" para este patamar
    resposta_dominante = contagem.most_common(1)[0][0]
    ocorrencias_dominante = contagem.most_common(1)[0][1]

    # Qualquer coisa que não seja a resposta dominante é um erro de reprodutibilidade
    taxa_erro = ((iteracoes_por_grupo - ocorrencias_dominante) / iteracoes_por_grupo) * 100

    resultados_estatisticos.append({
        "Carga": carga,
        "Respostas_Unicas": qtd_respostas_unicas,
        "Frequencia_Erro_%": taxa_erro,
        "Resposta_Dominante": resposta_dominante,
        "Todas_Variacoes": list(contagem.keys())
    })

df_resultados = pd.DataFrame(resultados_estatisticos)
print("\n✅ Teste de Consistência Interna Concluído!")

In [ ]:
# Célula 6 V4: Análise Semântica de Consistência Interna e Correlação
from sentence_transformers import SentenceTransformer, util

print("Carregando modelo de Embeddings (all-MiniLM-L6-v2)...")
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

print("\n" + "="*70)
print("RELATÓRIO DE SENSIBILIDADE E DRIFT SEMÂNTICO (CONSISTÊNCIA INTERNA)")
print("="*70)

for res in resultados_estatisticos:
    carga = res['Carga']
    frequencia_erro = res['Frequencia_Erro_%']
    respostas_unicas = res['Respostas_Unicas']
    dominante = res['Resposta_Dominante']
    todas_variacoes = res['Todas_Variacoes']

    print(f"\nNível de Carga: {carga} req/simultâneas")
    print(f"Respostas Únicas: {respostas_unicas} | Frequência de Erro Interno: {frequencia_erro:.2f}%")

    # Se houver mais de 1 resposta única, o determinismo quebrou naquele patamar
    if respostas_unicas > 1:
        print(f"  [Moda/Dominante]: '{dominante}'")
        emb_dominante = modelo_emb.encode(dominante, convert_to_tensor=True)

        # Calcular o drift de todas as anomalias em relação à resposta dominante
        for var in todas_variacoes:
            if var != dominante:
                emb_var = modelo_emb.encode(var, convert_to_tensor=True)
                sim = util.cos_sim(emb_dominante, emb_var).item() * 100
                print(f"  [Anomalia]: '{var}'")
                print(f"  ↳ Similaridade: {sim:.2f}% | Drift Semântico: {100-sim:.2f}%")
    else:
        print("  -> Determinismo absoluto preservado (100% de consistência).")

# Cálculo do Coeficiente de Pearson (A prova de correlação Carga vs Falha)
correlacao = df_resultados['Carga'].corr(df_resultados['Frequencia_Erro_%'])

print("\n" + "-"*70)
print(f"COEFICIENTE DE CORRELAÇÃO (Pearson): {correlacao:.4f}")

if correlacao > 0.7:
    interpretacao = "Forte correlação positiva entre Carga e Não-Determinismo."
elif correlacao > 0.3:
    interpretacao = "Correlação moderada detectada (comum em sistemas complexos de hardware)."
else:
    interpretacao = "Correlação fraca. O ruído parece não escalar linearmente com a carga."

print(f"Interpretação: {interpretacao}")
print("-" * 70)

In [ ]:
# Célula 7: Visualização Acadêmica e Exportação de Dados (Padrão Artigo)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

x = df_resultados['Carga']
y = df_resultados['Frequencia_Erro_%']

plt.scatter(x, y, color='#1f77b4', alpha=0.8, edgecolors='w', s=70, label='Ocorrência de Erro (Medição Real)')

# 3. Cálculo da Linha de Tendência (Regressão Polinomial para capturar a curva S)
z = np.polyfit(x, y, 3) # Polinômio de grau 3 para suavizar a curva de degradação
p = np.poly1d(z)

# Gerando a linha suave
x_linha = np.linspace(min(x), max(x), 100)
plt.plot(x_linha, p(x_linha), color='#d62728', linestyle='--', linewidth=2, label='Curva de Tendência de Degradação')

# 4. Marcando o Ponto de Ruptura (Tipping Point ~ 50 requisições)
plt.axvline(x=50, color='gray', linestyle=':', linewidth=1.5, label='Ponto de Ruptura Estimado (~50 req)')

# 5. Formatação Títulos e Rótulos
plt.title('Degradação do Determinismo Semântico sob Carga Concorrente (vLLM / A100)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Densidade do Lote Dinâmico (Requisições Simultâneas)', fontsize=12, labelpad=10)
plt.ylabel('Frequência de Erro Interno (%)', fontsize=12, labelpad=10)

# Travando os eixos para fazer sentido percentual
plt.ylim(-5, 105)
plt.xlim(0, 260)

# Posicionando a legenda
plt.legend(loc='upper left', fontsize=10, frameon=True, shadow=True)

# 6. Salvando a imagem em 300 DPI
plt.savefig('Grafico_TCC_Degradacao_Alta_Resolucao.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. Exportação dos Dados Brutos para Excel
df_resultados.to_excel('Resultados_MonteCarlo_TCC.xlsx', index=False)

print("\n" + "="*70)
print("SUCESSO! Arquivos gerados:")
print("1. Grafico_TCC_Degradacao_Alta_Resolucao.png ")
print("2. Resultados_MonteCarlo_TCC.xlsx ")
print("="*70)